In [7]:
#Python
from pyrosetta import *
from pyrosetta.rosetta import *
from pyrosetta.teaching import *
from pyrosetta.toolbox import *

#Core Includes
#from rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.core.kinematics import FoldTree
from rosetta.core.pack.task import TaskFactory
from rosetta.core.pack.task import operation
from rosetta.core.simple_metrics import metrics
from rosetta.core.select import residue_selector as selections
from rosetta.core import select
from rosetta.core.select.movemap import *

#Protocol Includes
from rosetta.protocols import minimization_packing as pack_min
from rosetta.protocols import relax as rel
from rosetta.protocols.antibody.residue_selector import CDRResidueSelector
from rosetta.protocols.antibody import *
from rosetta.protocols.loops import *
from rosetta.protocols.relax import FastRelax

#double check that everything is outputting properly in the notebook
print('Checkpoint')


Checkpoint


In [9]:
#initializes pyRosetta with the necessary flags and specifications
# Initializes PyRosetta by splitting arguments properly and muting all junk
init(options=[
    '-use_input_sc',
    '-input_ab_scheme', 'AHo_Scheme',
    '-ignore_unrecognized_res',
    '-ignore_zero_occupancy', 'false',
    '-load_PDB_components', 'false',
    '-relax:default_repeats', '2',
    '-no_fconfig',
    '-mute', 'all'  
])

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python313.Release 2026.26+release.f8a8eab4344af49c25e1e5db84ce25fec05eea27 2026-06-25T18:00:21] retrieved from: http://www.pyrosetta.org


In [10]:
# Rename this path to wherever you have saved your pdb file
pose = pose_from_pdb('7K18.pdb')

In [11]:
#the original PDB file will have a bunch of stuff (like extra chains of the protein) so we remove that here

print(os.getcwd())
print(pose.sequence())
print(pose.residue(958).name())
print(len(pose.sequence()))
#we want residues 1-111 in our truncated version, just chain A

/home/jimmy/coding/Toxin-Engineering-
VRRAAVKILVHSLFSMLIMCTILTNCVFMAQHDPPPWTKYVEYTFTAIYTFESLVKILARGFCLHAFTFLRDPWNWLDFSVIVMAYTTEFVDGNVSALRTFRVLRALKTISVISGLKTIVGALIQSVKKLADVMVLTVFCLSVFALIGLQLFMGNLRHKCVRNFTELNGTNGSVEASLDVYLNDPANYLLKNGTTDVLLCGNSSDAGTCPEGYRCLKAGENPDHGYTSFDSFAWAFLALFRLMTQDCWERLYQQTLRSAGKIYMIFFMLVIFLGSFYLVNLILAVVAMAYEEQNQATECCPLWMSIKQKVKFVVMDPFADLTITMCIVLNTLFMALEHYNMTAEFEEMLQVGNLVFTGIFTAEMTFKIIALDPYYYFQQGWNIFDSIIVILSLMELGSVLRSFRLLRVFKLAKSWPTLNTLIKIIGNSVGALGNLTLVLAIIVFIFAVVGMQLFGKNYSELRHRISDSGLLPRWHMMDFFHAFLIIFRILCGEWIETMWDCMEVSGQSLCLLVFLLVMVIGNLVVLNLFLALLLSSFGKVWWRLRKTCYRIVEHSWFETFIIFMILLSSGALAFEDIYLEERKTIKVLLEYADKMFTYVFVLEMLLKWVAYGFKKYFTNAWCWLDFLIVDVSLVSLVANTLGFAEMGPIKSLRTLRALRPLRALSRFEGMRVVVNALVGAIPSIMNVLLVCLIFWLIFSIMGVNLFAGKFGRCINQTEGDLPLNYTIVNNKSECESFNVTGELYWTKVKVNFDNVGAGYLALLQVATFKGWMDIMYAAVDSRGYEEQPQWEDNLYMYIYFVVFIIFGSFFTLNLFIGVIIDNFNQQKKKLGGQDIFMTEEQKKYYNAMKKLGSKKPQKPIPRPLNKYQGFIFDIVTKQAFDVTIMFLICLNMVTMMVETDDQSPEKVNILAKINLLFVAIFTGECIVKMAALRHYYFTNSWNIFDFVVVILSIVGTVLSDII

In [ ]:

# Shorten sequence

trunc_pose = Pose()
trunc_pose.assign(pose)
pyrosetta.rosetta.protocols.grafting.delete_region(trunc_pose,1,899)
print(trunc_pose.sequence())



DDQSPEKVNILAKINLLFVAIFTGECIVKMAALRHYYFTNSWNIFDFVVVILSIVGTVLSDIIQKYFFSPTLFRVIRLARIGRILRLIRGAKGIRTLLFALMMSLPALFNIGLLLFLVMFIYSIFGMANFAYVKWEAGIDDMFNFQTFANSMLCLFQITTSAGWDGLLSPILNTGPPYCDPNLPNSNGSRGNCGSPAVGILFFTTYIIISFLIVVNMYIAIILENFSVAVRDGYIAQPENCVYHCFPGSSGCDTLCKEKGGTSGHCGFKVGHGLACWCNALPDNVGIIVEGEKCHS


In [15]:
#relax the structure

def relax_structure(pose_to_relax, output_name):

    # Create new pose and assign pose to relax to it
    testPose = Pose()
    testPose.assign(pose_to_relax)
    print(testPose)

    # Set up relax parameter
    scorefxn = get_fa_scorefxn()
    relax = rosetta.protocols.relax.FastRelax()
    relax.set_scorefxn(scorefxn)
    relax.constrain_relax_to_start_coords(True)
    print(relax)
    relax.apply(testPose)

    #rename to your desired relaxed structure name
    testPose.dump_pdb(output_name)

In [16]:
relax_structure(trunc_pose, 'relax.pdb')

PDB file name: 7K18.pdb
Total residues: 296
Sequence: DDQSPEKVNILAKINLLFVAIFTGECIVKMAALRHYYFTNSWNIFDFVVVILSIVGTVLSDIIQKYFFSPTLFRVIRLARIGRILRLIRGAKGIRTLLFALMMSLPALFNIGLLLFLVMFIYSIFGMANFAYVKWEAGIDDMFNFQTFANSMLCLFQITTSAGWDGLLSPILNTGPPYCDPNLPNSNGSRGNCGSPAVGILFFTTYIIISFLIVVNMYIAIILENFSVAVRDGYIAQPENCVYHCFPGSSGCDTLCKEKGGTSGHCGFKVGHGLACWCNALPDNVGIIVEGEKCHS
Fold tree:
FOLD_TREE  EDGE 1 229 -1  EDGE 1 230 1  EDGE 230 296 -1 


In [17]:
#WRITE FUNCTION FOR MUTATION
import pyrosetta
from pyrosetta import pose_from_pdb, get_score_function
import os
import random

def pack(pose, posi, amino, scorefxn):

    #Set Reference Pose
    RMSD_calc = pyrosetta.rosetta.core.simple_metrics.metrics.RMSDMetric(pose)
    
    
    # Select Mutate Position
    mut_posi = pyrosetta.rosetta.core.select.residue_selector.ResidueIndexSelector()
    mut_posi.set_index(posi)
    #print(pyrosetta.rosetta.core.select.get_residues_from_subset(mut_posi.apply(pose)))

    # Select Neighbor Position
    nbr_selector = pyrosetta.rosetta.core.select.residue_selector.NeighborhoodResidueSelector()
    nbr_selector.set_focus_selector(mut_posi)
    nbr_selector.set_include_focus_in_subset(True)
    #print(pyrosetta.rosetta.core.select.get_residues_from_subset(nbr_selector.apply(pose)))
    
    # Select No Design Area
    not_design = pyrosetta.rosetta.core.select.residue_selector.NotResidueSelector(mut_posi)
    #print(pyrosetta.rosetta.core.select.get_residues_from_subset(not_design.apply(pose)))

    # The task factory accepts all the task operations
    tf = pyrosetta.rosetta.core.pack.task.TaskFactory()

    # These are pretty standard
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.InitializeFromCommandline())
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.IncludeCurrent())
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.NoRepackDisulfides())

    # Disable Packing
    prevent_repacking_rlt = pyrosetta.rosetta.core.pack.task.operation.PreventRepackingRLT()
    #True indicates here that we are flipping the selection.  So that we are turning off everything but the CDR and its neighbors.
    prevent_subset_repacking = pyrosetta.rosetta.core.pack.task.operation.OperateOnResidueSubset(prevent_repacking_rlt, nbr_selector, True )
    tf.push_back(prevent_subset_repacking)

    # Disable design
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.OperateOnResidueSubset(
        pyrosetta.rosetta.core.pack.task.operation.RestrictToRepackingRLT(),not_design))

    # Enable design
    aa_to_design = pyrosetta.rosetta.core.pack.task.operation.RestrictAbsentCanonicalAASRLT()
    aa_to_design.aas_to_keep(amino)
    tf.push_back(pyrosetta.rosetta.core.pack.task.operation.OperateOnResidueSubset(aa_to_design, mut_posi))
    
    # Create Packer
    packer = pyrosetta.rosetta.protocols.minimization_packing.PackRotamersMover()
    packer.task_factory(tf)
    # print(tf.create_task_and_apply_taskoperations(pose))


    
    #Perform The Move
    if not os.getenv("DEBUG"):
      packer.apply(pose)

In [18]:
#Load the previously-relaxed pose.
#grab your initial structure again

def perform_mutation(pdb, pos, amino):
    relaxPose = pose_from_pdb(pdb)
    # Clone it
    original = relaxPose.clone()
    scorefxn = get_score_function()


    # #Input the residue number that you wish to mutate and the 1-letter code 
    # #If you are substituting multiple, you can just have them listed separately as exampled below
    pack(relaxPose, pos, amino, scorefxn)
    # #pack(relaxPose, 81, 'D', scorefxn)
    # print("\nNew Energy:", scorefxn(relaxPose),"\n")

    original_aa = str(original.residue(pos))
    mutated_aa = relaxPose.residue(pos)

    # #SAVE THE NEW PDB FILE HERE:
    path = f'7K18_{pos}{amino}.pdb'
    relaxPose.dump_pdb(path)


    # #Set relaxPose back to original 
    # relaxPose = original.clone()

    print()
    print('-' * 50)
    print(f'Mutated structure succesfully saved as {path}')
    print(f'Successfully mutated {original.residue(pos).name()} at position {pos} to {relaxPose.residue(pos).name()}')
    print(f'Orginal Energy {scorefxn(original)}; New energy: {scorefxn(relaxPose)}')
    print('-' * 50)
    print('\n')


#Figure out how to minimize the structure or select the rotamers of residues in the region that produces the smallest energy/strain

In [46]:
perform_mutation('test.relax.pdb', 59, 'M')

core.import_pose.import_pose: File 'test.relax.pdb' automatically determined to be of type PDB from contents.
core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue ALA:CtermProteinFull 100
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
protocols.minimization_packing.PackRotamersMover: [ WARNING ] undefined ScoreFunction -- creating a default one
core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.pack.task: Packer task: initialize from command line()
core.pack.pack_rotamers: built 122 rotamers at 12 positions.
core.pack.pack_rotamers: Requesting all available threads for interaction graph computation.
core.pack.interaction_graph.interaction_graph_factory: Instantiating PDInteractionGraph
core.pack.rotamer_set.RotamerSets: Completed interaction graph pre-calculation in 1 available threads (1 had been requested).

--------------------------------------------------
Mutated structure succesfully saved as 7K18_59M.pdb
Successfully mutated LEU a

In [ ]:
from pymol import cmd
from Ipython.display import Image, display
cmd.load('7K18_59M.pdb')
